# Bro SuperPoint Transformer - Visualization
Load the latest bridge model and visualize predictions without ConfigUI

In [1]:
import os
import sys
from pathlib import Path

import hydra
import numpy as np
import pandas as pd
import torch

# Add project to path
sys.path.insert(0, '/cluster/home/larshfle/superpoint_transformer_new')

from src.utils import init_config
from src.transforms import *
from src.data import *

print("Imports successful!")

Imports successful!


## 0. Look at a raw bridge tile before preprocessing and training

In [ ]:
from src.datasets.bro import read_bro_las
from src.datasets.bro_config import BRO_NUM_CLASSES, CLASS_COLORS, CLASS_NAMES

# Mini test tiles:
# "bro_000082", "bro_001709", "bro_001409"

raw_preview_path = '/cluster/home/larshfle/superpoint_transformer_new/export_bro_000166_predicted.las'
raw_data = read_bro_las(raw_preview_path)
raw_data.show(
    class_names=CLASS_NAMES,
    class_colors=CLASS_COLORS,
    num_classes=BRO_NUM_CLASSES,
    max_points=100000,
)


FileNotFoundError: [Errno 2] No such file or directory: '/cluster/home/larshfle/superpoint_transformer_new/export_bro_000166_prediction.las'

## 1. Setup paths and config

In [ ]:
# Settings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
split = 'test'  # Choose: 'train', 'val', or 'test'
tile = 'bro_000166'
ckpt_path = '/cluster/home/larshfle/superpoint_transformer_new/logs/train/runs/2026-04-27_14-11-56/checkpoints/last.ckpt'
data_dir = '/cluster/home/larshfle/datasets/bro'

print(f"Device: {device}")
print(f"Split: {split}")
print(f"Tile: {tile}")
print(f"Checkpoint: {ckpt_path}")
print(f"Data directory: {data_dir}")

# Verify checkpoint exists
assert os.path.exists(ckpt_path), f"Checkpoint not found at {ckpt_path}"
print("Checkpoint found!")

## 2. Load config and instantiate datamodule

In [ ]:
# Parse config with overrides
cfg = init_config(overrides=[
    'experiment=semantic/bro',
    f'ckpt_path={ckpt_path}',
    'datamodule.mini=false',
    'datamodule.load_full_res_idx=true',
])

# Override data_dir if needed
cfg.datamodule.data_dir = data_dir

print("Config loaded!")
print("Experiment: semantic/bro")
print(f"Model: {cfg.model._target_}")

In [ ]:
# Instantiate datamodule
datamodule = hydra.utils.instantiate(cfg.datamodule)
datamodule.prepare_data()
datamodule.setup()

# Get dataset
if split == 'train':
    dataset = datamodule.train_dataset
elif split == 'val':
    dataset = datamodule.val_dataset
elif split == 'test':
    dataset = datamodule.test_dataset
else:
    raise ValueError(f"Unknown split '{split}'")

sample_idx = dataset.cloud_ids.index(tile)

print(f"Datamodule loaded! Split: {split}")
print(f"Selected tile: {tile} ({sample_idx=})")
dataset.print_classes()

## 3. Load model

In [ ]:
# Instantiate model
model = hydra.utils.instantiate(cfg.model)

# Load checkpoint
load_kwargs = {}
pretrained_cnn_ckpt_path = cfg.datamodule.get("pretrained_cnn_ckpt_path", None)
if pretrained_cnn_ckpt_path is not None:
    load_kwargs["pretrained_cnn_ckpt_path"] = pretrained_cnn_ckpt_path

model = model._load_from_checkpoint(
    cfg.ckpt_path,
    **load_kwargs
)

# Move to device and eval mode
model = model.eval().to(device)

print("Model loaded from checkpoint!")
print(f"Device: {device}")

## 4. Run inference on first sample

In [ ]:
# Enable feature storage for visualization
model.net.store_features = True

# Load selected sample
print(f"Loading {tile}...")
nag = dataset[sample_idx]
raw_path = os.path.join(data_dir, 'raw', split, f'{tile}.las')

# Apply on-device transforms
nag = dataset.on_device_transform(nag.to(device))

print("Sample loaded!")
print(f"  - Raw path: {raw_path}")
print(f"  - Level 0 points: {nag[0].pos.shape[0]}")
print(f"  - Level 1 superpoints: {nag[1].pos.shape[0]}")

In [ ]:
# Run inference
print("Running inference...")
with torch.no_grad():
    output = model(nag)

print("Inference complete!")

# Get voxel-wise predictions
nag[0].semantic_pred = output.voxel_semantic_pred(super_index=nag[0].super_index)

# Get object/panoptic predictions if applicable
if hasattr(output, 'voxel_panoptic_pred'):
    vox_y, vox_index, vox_obj_pred = output.voxel_panoptic_pred(super_index=nag[0].super_index)
    nag[0].obj_pred = vox_obj_pred
    print("  - Panoptic predictions added")

print(f"Predictions shape: {nag[0].semantic_pred.shape}")

## 5. Bro labels and predictions

In [ ]:
from src.datasets.bro import read_bro_las
from src.datasets.bro_config import BRO_NUM_CLASSES, CLASS_NAMES

# Recover full-resolution predictions and compare them to the remapped bridge labels.
raw_semantic_pred = output.full_res_semantic_pred(
    super_index_level0_to_level1=nag[0].super_index,
    sub_level0_to_raw=nag[0].sub,
).cpu()

raw_data = read_bro_las(raw_path)
raw_labels = raw_data.y.cpu()
valid_mask = raw_labels != BRO_NUM_CLASSES

analysis_df = pd.DataFrame({
    'target_id': raw_labels[valid_mask].numpy(),
    'pred_id': raw_semantic_pred[valid_mask].numpy(),
})

class_name_by_id = {i: name for i, name in enumerate(CLASS_NAMES[:BRO_NUM_CLASSES])}
analysis_df['target'] = analysis_df['target_id'].map(class_name_by_id)
analysis_df['prediction'] = analysis_df['pred_id'].map(class_name_by_id)

confusion_table = pd.crosstab(
    analysis_df['target'],
    analysis_df['prediction'],
    rownames=['target'],
    colnames=['prediction'],
).reindex(
    index=CLASS_NAMES[:BRO_NUM_CLASSES],
    columns=CLASS_NAMES[:BRO_NUM_CLASSES],
    fill_value=0,
)

confusion_table['Total'] = confusion_table.sum(axis=1)
confusion_table['Bridge prediction %'] = (
    100 * confusion_table.get('bridge', 0) / confusion_table['Total'].replace(0, np.nan)
).round(2)

display(confusion_table)

bridge_target = analysis_df['target'] == 'bridge'
bridge_pred = analysis_df['prediction'] == 'bridge'

bridge_recall = 100 * (bridge_target & bridge_pred).sum() / bridge_target.sum() if bridge_target.any() else float('nan')
bridge_precision = 100 * (bridge_target & bridge_pred).sum() / bridge_pred.sum() if bridge_pred.any() else float('nan')

print(f"Trainable full-res points: {len(analysis_df):,}")
print(f"Ignored full-res points: {(~valid_mask).sum().item():,}")
print(f"Bridge recall: {bridge_recall:.2f}%")
print(f"Bridge precision: {bridge_precision:.2f}%")

## 6. Visualize full tile

In [ ]:
# Visualize hierarchical partition
nag.show(
    class_names=dataset.class_names,
    class_colors=dataset.class_colors,
    stuff_classes=dataset.stuff_classes,
    num_classes=dataset.num_classes,
    max_points=100000
)

## 7. Visualize a cropped region

In [ ]:
# Define region
center = nag[0].pos.mean(dim=0).view(1, -1)
radius = 25

print(f"Center: {center.squeeze().tolist()}")
print(f"Radius: {radius}")

# Visualize
nag.show(
    radius=radius,
    center=center,
    class_names=dataset.class_names,
    class_colors=dataset.class_colors,
    stuff_classes=dataset.stuff_classes,
    num_classes=dataset.num_classes,
    max_points=100000
)

## 8. Visualize graph structure

In [ ]:
# Visualize with superpoint centroids and edges
nag.show(
    radius=radius,
    center=center,
    class_names=dataset.class_names,
    class_colors=dataset.class_colors,
    stuff_classes=dataset.stuff_classes,
    num_classes=dataset.num_classes,
    max_points=100000,
    centroids=True,
    h_edge=True,
    h_edge_width=2
)

## 9. Export to HTML

In [ ]:
# Export visualization
output_path = '/cluster/home/larshfle/superpoint_transformer_new/bro_visualization.html'

nag.show(
    figsize=1600,
    radius=radius,
    center=center,
    class_names=dataset.class_names,
    class_colors=dataset.class_colors,
    stuff_classes=dataset.stuff_classes,
    num_classes=dataset.num_classes,
    max_points=100000,
    title='Bro SPT Predictions (2026-04-13_13-56-46 last.ckpt)',
    path=output_path
)

print(f"Visualization exported to: {output_path}")

In [ ]:
## Per-tile analysis on full test set
from src.datasets.bro import read_bro_las
from src.datasets.bro_config import BRO_NUM_CLASSES

results = []

for i, tile_id in enumerate(dataset.cloud_ids):
    try:
        nag_i = dataset[i]
        nag_i = dataset.on_device_transform(nag_i.to(device))

        with torch.no_grad():
            out_i = model(nag_i)

        raw_pred = out_i.full_res_semantic_pred(
            super_index_level0_to_level1=nag_i[0].super_index,
            sub_level0_to_raw=nag_i[0].sub,
        ).cpu()

        raw_path_i = os.path.join(data_dir, 'raw', split, f'{tile_id}.las')
        raw_labels_i = read_bro_las(raw_path_i).y.cpu()
        valid = raw_labels_i != BRO_NUM_CLASSES

        gt = raw_labels_i[valid]
        pred = raw_pred[valid]

        n_bridge_gt = (gt == 1).sum().item()
        tp = ((gt == 1) & (pred == 1)).sum().item()
        fp = ((gt == 0) & (pred == 1)).sum().item()
        fn = ((gt == 1) & (pred == 0)).sum().item()

        recall    = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
        precision = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
        iou       = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else float('nan')

        results.append({
            'tile': tile_id,
            'n_points': valid.sum().item(),
            'n_bridge_gt': n_bridge_gt,
            'bridge_pct': round(100 * n_bridge_gt / valid.sum().item(), 3) if valid.sum() > 0 else 0,
            'recall': round(recall, 3),
            'precision': round(precision, 3),
            'iou': round(iou, 3),
        })

        # Fri GPU-minne mellom tiles
        del nag_i, out_i, raw_pred
        torch.cuda.empty_cache()

        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(dataset.cloud_ids)} done...")

    except Exception as e:
        print(f"  ERROR on {tile_id}: {e}")
        results.append({'tile': tile_id, 'iou': float('nan')})

df = pd.DataFrame(results).sort_values('iou')
print(f"\n{'='*70}")
print(f"Tiles: {len(df)}  |  Mean IoU: {df['iou'].mean():.3f}  |  Median IoU: {df['iou'].median():.3f}")
print(f"{'='*70}\n")
print("--- 20 verste tiles ---")
print(df.head(20)[['tile','n_bridge_gt','bridge_pct','recall','precision','iou']].to_string(index=False))
print("\n--- 10 beste tiles ---")
print(df.tail(10)[['tile','n_bridge_gt','bridge_pct','recall','precision','iou']].to_string(index=False))

In [ ]:
## 10. Eksporter ground truth LAS
import laspy
import numpy as np

pos_np = (nag[0].pos + nag[0].pos_offset.to(nag[0].pos.device)).cpu().numpy().astype(np.float64)
gt_labels = nag[0].y.cpu().numpy()

colors = np.array(dataset.class_colors, dtype=np.uint16)
gt_rgb = colors[np.clip(gt_labels, 0, len(colors) - 1)] * 256  # 8-bit -> 16-bit

header = laspy.LasHeader(point_format=2, version='1.2')
las_gt = laspy.LasData(header=header)
las_gt.x = pos_np[:, 0]
las_gt.y = pos_np[:, 1]
las_gt.z = pos_np[:, 2]
las_gt.red   = gt_rgb[:, 0]
las_gt.green = gt_rgb[:, 1]
las_gt.blue  = gt_rgb[:, 2]
las_gt.classification = gt_labels.astype(np.uint8)

out_gt = f'/cluster/home/larshfle/superpoint_transformer_new/export_{tile}_groundtruth.las'
las_gt.write(out_gt)
print(f'Ground truth LAS lagret: {out_gt}  ({len(pos_np):,} punkter)')

In [ ]:
## 10. Eksporter ground truth LAS
import laspy
import numpy as np

pos_np = (nag[0].pos + nag[0].pos_offset.to(nag[0].pos.device)).cpu().numpy().astype(np.float64)
gt_labels = nag[0].y.cpu().numpy()

# Sikre at colors alltid er (C, 3) int-array
colors = np.array([list(c) for c in dataset.class_colors], dtype=np.uint16)
idx = np.clip(gt_labels, 0, len(colors) - 1)
gt_r = (colors[idx, 0] * 256).astype(np.uint16)
gt_g = (colors[idx, 1] * 256).astype(np.uint16)
gt_b = (colors[idx, 2] * 256).astype(np.uint16)

header = laspy.LasHeader(point_format=2, version='1.2')
las_gt = laspy.LasData(header=header)
las_gt.x = pos_np[:, 0]
las_gt.y = pos_np[:, 1]
las_gt.z = pos_np[:, 2]
las_gt.red   = gt_r
las_gt.green = gt_g
las_gt.blue  = gt_b
las_gt.classification = gt_labels.astype(np.uint8)

out_gt = f'/cluster/home/larshfle/superpoint_transformer_new/export_{tile}_groundtruth.las'
las_gt.write(out_gt)
print(f'Ground truth LAS lagret: {out_gt}  ({len(pos_np):,} punkter)')

In [ ]:
## 11. Eksporter predikert LAS
pred_labels = nag[0].semantic_pred.cpu().numpy()

pred_r = (colors[np.clip(pred_labels, 0, len(colors) - 1), 0] * 256).astype(np.uint16)
pred_g = (colors[np.clip(pred_labels, 0, len(colors) - 1), 1] * 256).astype(np.uint16)
pred_b = (colors[np.clip(pred_labels, 0, len(colors) - 1), 2] * 256).astype(np.uint16)

header = laspy.LasHeader(point_format=2, version='1.2')
las_pred = laspy.LasData(header=header)
las_pred.x = pos_np[:, 0]
las_pred.y = pos_np[:, 1]
las_pred.z = pos_np[:, 2]
las_pred.red   = pred_r
las_pred.green = pred_g
las_pred.blue  = pred_b
las_pred.classification = pred_labels.astype(np.uint8)

out_pred = f'/cluster/home/larshfle/superpoint_transformer_new/export_{tile}_predicted.las'
las_pred.write(out_pred)
print(f'Predikert LAS lagret: {out_pred}  ({len(pos_np):,} punkter)')